In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from src.preprocessing.geometry import load_pad_geometry
from src.preprocessing.pipeline import preprocess

# ── Set these to real detector files on your machine ──────────────────────────────────────────────
DETECTOR_FILES: dict[str, str] = {
    "AGIPD":        "/path/to/agipd_frame.h5",
    "JUNGFRAU_4M":  "/path/to/jungfrau_frame.cxi",
    "ePix10k":      "/path/to/epix_frame.h5",
    "Eiger4M":      "/path/to/eiger_frame.h5",
}

WINDOWS = [3, 9, 15, 31]
OUT_DIR = Path("docs/figures/lcn_ablation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import h5py


def load_panel_data(detector: str, path: str) -> tuple:
    """Return (panel_data_list, pads) for one detector frame."""
    pads = load_pad_geometry(detector)
    with h5py.File(path, "r") as f:
        raw = f["entry/data/data"][()]  # shape depends on detector
    if raw.ndim == 2:
        panel_data = [raw[i * pads[0].n_ss:(i + 1) * pads[0].n_ss, :]
                      for i in range(len(pads))] if len(pads) > 1 else [raw]
    elif raw.ndim == 3:
        panel_data = [raw[i] for i in range(raw.shape[0])]
    else:
        raise ValueError(f"Unexpected raw ndim {raw.ndim} for {detector}")
    return panel_data, pads

In [ ]:
fig, axes = plt.subplots(
    len(DETECTOR_FILES), len(WINDOWS),
    figsize=(4 * len(WINDOWS), 3 * len(DETECTOR_FILES)),
)

for row_idx, (detector, fpath) in enumerate(DETECTOR_FILES.items()):
    panel_data, pads = load_panel_data(detector, fpath)
    images = [preprocess(panel_data, pads, lcn_window=w) for w in WINDOWS]

    vmin = min(img.min() for img in images)
    vmax = max(img.max() for img in images)

    for col_idx, (w, img) in enumerate(zip(WINDOWS, images)):
        ax = axes[row_idx, col_idx]
        ax.imshow(img, cmap="viridis", vmin=vmin, vmax=vmax, origin="lower")
        ax.set_title(f"window={w}", fontsize=9)
        if col_idx == 0:
            ax.set_ylabel(detector, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle("LCN Window Ablation — all four detectors", fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / "all_detectors_lcn_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved all_detectors_lcn_comparison.png")

In [ ]:
for detector, fpath in DETECTOR_FILES.items():
    panel_data, pads = load_panel_data(detector, fpath)
    fig, axes = plt.subplots(1, len(WINDOWS), figsize=(4 * len(WINDOWS), 3))
    images = [preprocess(panel_data, pads, lcn_window=w) for w in WINDOWS]
    vmin = min(img.min() for img in images)
    vmax = max(img.max() for img in images)
    for ax, w, img in zip(axes, WINDOWS, images):
        ax.imshow(img, cmap="viridis", vmin=vmin, vmax=vmax, origin="lower")
        ax.set_title(f"window={w}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(detector, fontsize=11)
    plt.tight_layout()
    out = OUT_DIR / f"{detector}_lcn_comparison.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved {out}")

## Decision

After visual inspection:

- **Chosen window size:** _fill in after review_
- **Rationale:** _fill in after review_
- **Action:** Update `LCN_WINDOW_DEFAULT` in `src/preprocessing/normalize.py` if not 9.